# Egemen — Turkish SLM, From Scratch

Builds and pre-trains a decoder-only transformer **entirely from scratch** in PyTorch on the Turkish corpus.

**Architecture** (LLaMA-style):
- RMSNorm (pre-norm)
- Rotary Position Embeddings (RoPE)
- SwiGLU feed-forward
- Causal self-attention

**Default — Egemen-Small (~117M params):** 12 layers, 768 hidden, 12 heads, 1024 ctx  
**Target platform:** Kaggle free T4 (16 GB VRAM) — ~2 hrs/epoch

## 0 · GPU Check

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout
      or 'No GPU — enable T4 in Kaggle Settings > Accelerator')
import torch
print(f"PyTorch {torch.__version__} | device: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1 · Install Dependencies

In [ ]:
%%capture
!pip install -q tokenizers==0.21.1 huggingface_hub requests tqdm

## 2 · ntfy.sh Notifications

In [ ]:
import requests as _req

NTFY_URL = "https://ntfy.sh/ozan"

def notify(message: str, title: str = "Egemen", priority: str = "default", tags: str = ""):
    """Send a push notification to ntfy.sh/ozan. Fails silently so training never crashes."""
    try:
        headers = {"Title": title, "Priority": priority}
        if tags:
            headers["Tags"] = tags
        _req.post(NTFY_URL, data=message.encode("utf-8"), headers=headers, timeout=5)
    except Exception:
        pass

# Test it
notify("Notebook loaded successfully.", tags="white_check_mark")
print("ntfy.sh ready — check ntfy.sh/ozan")

## 3 · Config

In [ ]:
from dataclasses import dataclass

@dataclass
class EgemenConfig:
    # architecture
    vocab_size:   int   = 32_000
    n_layers:     int   = 12
    n_heads:      int   = 12
    d_model:      int   = 768
    max_seq_len:  int   = 1024
    dropout:      float = 0.1
    bias:         bool  = False
    # training
    batch_size:      int   = 8
    grad_accum:      int   = 4
    learning_rate:   float = 3e-4
    weight_decay:    float = 0.1
    grad_clip:       float = 1.0
    num_epochs:      int   = 3
    warmup_steps:    int   = 1000
    eval_interval:   int   = 500
    save_interval:   int   = 1000
    log_interval:    int   = 10
    # tokenizer
    tokenizer_vocab_size: int = 32_000
    # paths
    tokenizer_path: str = "./egemen-tokenizer"
    checkpoint_dir: str = "./egemen-checkpoints"
    hf_repo:        str = "smartlizardpy/egemen"

    @property
    def d_ff(self) -> int:
        raw = int(8 / 3 * self.d_model)
        return (raw + 255) // 256 * 256

    @property
    def head_dim(self) -> int:
        return self.d_model // self.n_heads

cfg = EgemenConfig()
embed  = cfg.vocab_size * cfg.d_model
attn   = cfg.n_layers * 4 * cfg.d_model * cfg.d_model
ff     = cfg.n_layers * 3 * cfg.d_model * cfg.d_ff
total  = embed + attn + ff
print(f"Egemen-Small: ~{total/1e6:.0f}M parameters | d_ff={cfg.d_ff}")

## 4 · Fetch Training Data from GitHub

In [ ]:
import os
from pathlib import Path
from tqdm.auto import tqdm

RAW = "https://raw.githubusercontent.com/smartlizardpy/Egemen/main"
API = "https://api.github.com/repos/smartlizardpy/Egemen/git/trees/main?recursive=1"

os.makedirs("data/news", exist_ok=True)

def download(url, dest):
    if Path(dest).exists(): return
    r = _req.get(url, timeout=60)
    r.raise_for_status()
    Path(dest).write_bytes(r.content)

notify("Downloading training data from GitHub...", tags="arrow_down")

download(f"{RAW}/scraper/data/people.txt", "data/people.txt")

tree = _req.get(API, timeout=30).json()
news_paths = [i["path"] for i in tree.get("tree", [])
              if i["path"].startswith("scraper/data/news/") and i["path"].endswith(".txt")]
print(f"Found {len(news_paths)} news files")

for path in tqdm(news_paths):
    download(f"{RAW}/{path}", f"data/news/{path.replace('/', '_')}")

notify(f"Data ready: {len(news_paths)} news files + people.txt", tags="white_check_mark")
print("Done.")

## 5 · Train Turkish BPE Tokenizer

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers

def iter_texts():
    for path in Path("data").rglob("*.txt"):
        text = path.read_text(encoding="utf-8", errors="ignore")
        if text.startswith("SOURCE:"):
            text = text.split("\n\n", 1)[-1]
        text = "\n".join(l for l in text.splitlines()
                         if not l.startswith("===") and not l.startswith("---"))
        if len(text.strip()) > 50:
            yield text

os.makedirs(cfg.tokenizer_path, exist_ok=True)
tok_file = cfg.tokenizer_path + "/tokenizer.json"

if Path(tok_file).exists():
    print("Tokenizer already trained.")
else:
    notify("Training BPE tokenizer...", tags="pencil")
    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.normalizer    = normalizers.NFKC()
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder       = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=cfg.tokenizer_vocab_size,
        special_tokens=["<unk>", "<s>", "</s>", "<pad>", "<mask>"],
        min_frequency=2, show_progress=True,
    )
    tokenizer.train_from_iterator(iter_texts(), trainer=trainer)
    tokenizer.save(tok_file)
    notify("Tokenizer trained!", tags="white_check_mark")

tokenizer = Tokenizer.from_file(tok_file)
BOS_ID = tokenizer.token_to_id("<s>")
EOS_ID = tokenizer.token_to_id("</s>")
PAD_ID = tokenizer.token_to_id("<pad>")

enc = tokenizer.encode("Ankara Türkiye'nin başkentidir.")
print(f"Test tokens: {enc.tokens}")

## 6 · Tokenize & Cache Dataset

In [ ]:
import numpy as np

CACHE = "data/tokens.bin"

if Path(CACHE).exists():
    token_array = np.fromfile(CACHE, dtype=np.uint16)
    print(f"Loaded cached tokens: {len(token_array):,}")
else:
    notify("Tokenizing dataset (one-time, ~5-10 min)...", tags="hourglass")
    all_ids = []
    for path in tqdm(list(Path("data").rglob("*.txt"))):
        text = path.read_text(encoding="utf-8", errors="ignore")
        if text.startswith("SOURCE:"): text = text.split("\n\n", 1)[-1]
        text = "\n".join(l for l in text.splitlines()
                         if not l.startswith("===") and not l.startswith("---")).strip()
        if len(text) < 50: continue
        all_ids.extend([BOS_ID] + tokenizer.encode(text).ids + [EOS_ID])
    token_array = np.array(all_ids, dtype=np.uint16)
    token_array.tofile(CACHE)
    notify(f"Dataset tokenized: {len(token_array)/1e6:.1f}M tokens", tags="white_check_mark")

split      = int(len(token_array) * 0.98)
train_data = token_array[:split]
val_data   = token_array[split:]
print(f"Train: {len(train_data)/1e6:.1f}M | Val: {len(val_data)/1e6:.1f}M tokens")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class TokenDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data, self.seq_len = data, seq_len
    def __len__(self):
        return (len(self.data) - 1) // self.seq_len
    def __getitem__(self, idx):
        s = idx * self.seq_len
        c = self.data[s : s + self.seq_len + 1]
        return (torch.from_numpy(c[:-1].astype(np.int64)),
                torch.from_numpy(c[1:].astype(np.int64)))

train_loader = DataLoader(TokenDataset(train_data, cfg.max_seq_len),
                          batch_size=cfg.batch_size, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(TokenDataset(val_data, cfg.max_seq_len),
                          batch_size=cfg.batch_size, shuffle=False,
                          num_workers=2, pin_memory=True, drop_last=True)
print(f"Train batches: {len(train_loader):,} | Val batches: {len(val_loader):,}")

## 7 · Model Architecture

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import math

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.w, self.eps = nn.Parameter(torch.ones(d)), eps
    def forward(self, x):
        return self.w * x / x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len=4096, base=10_000):
        super().__init__()
        inv = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv, persistent=False)
        t   = torch.arange(max_seq_len).float()
        emb = torch.cat([torch.outer(t, inv)] * 2, dim=-1)
        self.register_buffer("cos", emb.cos()[None, None], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None], persistent=False)

    @staticmethod
    def rot(x):
        h = x.shape[-1] // 2
        return torch.cat([-x[..., h:], x[..., :h]], dim=-1)

    def forward(self, q, k):
        T = q.shape[2]
        c, s = self.cos[:, :, :T], self.sin[:, :, :T]
        return q * c + self.rot(q) * s, k * c + self.rot(k) * s

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.H, self.hd = cfg.n_heads, cfg.head_dim
        self.qkv  = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=cfg.bias)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model,     bias=cfg.bias)
        self.drop = nn.Dropout(cfg.dropout)
        self.rope  = RotaryEmbedding(self.hd, cfg.max_seq_len)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, -1)
        q, k, v = [t.view(B, T, self.H, self.hd).transpose(1, 2) for t in (q, k, v)]
        q, k    = self.rope(q, k)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                dropout_p=self.drop.p if self.training else 0.0)
        return self.proj(y.transpose(1, 2).contiguous().view(B, T, C))

class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.up   = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.drop(self.down(F.silu(self.gate(x)) * self.up(x)))

class EgemenBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n1, self.attn = RMSNorm(cfg.d_model), CausalSelfAttention(cfg)
        self.n2, self.ff   = RMSNorm(cfg.d_model), SwiGLU(cfg)
    def forward(self, x):
        x = x + self.attn(self.n1(x))
        x = x + self.ff(self.n2(x))
        return x

class Egemen(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg    = cfg
        self.embed  = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop   = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([EgemenBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm   = RMSNorm(cfg.d_model)
        self.head   = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.embed.weight  # weight tying
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, x, targets=None):
        x = self.drop(self.embed(x))
        for b in self.blocks: x = b(x)
        logits = self.head(self.norm(x))
        loss   = F.cross_entropy(logits.view(-1, self.cfg.vocab_size),
                                 targets.view(-1), ignore_index=PAD_ID) if targets is not None else None
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50):
        self.eval()
        for _ in range(max_new_tokens):
            ctx    = idx[:, -self.cfg.max_seq_len:]
            logits, _ = self(ctx)
            logits = logits[:, -1] / temperature
            if top_k:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            next_tok = torch.multinomial(logits.softmax(-1), 1)
            idx = torch.cat([idx, next_tok], dim=1)
            if next_tok.item() == EOS_ID: break
        return idx

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = Egemen(cfg).to(DEVICE)
print(f"Egemen | {model.num_params()/1e6:.1f}M params | {DEVICE}")

## 8 · Train

In [ ]:
import math, time

os.makedirs(cfg.checkpoint_dir, exist_ok=True)

decay   = [p for n, p in model.named_parameters() if p.ndim >= 2]
nodecay = [p for n, p in model.named_parameters() if p.ndim < 2]
optimizer = torch.optim.AdamW(
    [{"params": decay,   "weight_decay": cfg.weight_decay},
     {"params": nodecay, "weight_decay": 0.0}],
    lr=cfg.learning_rate, betas=(0.9, 0.95),
    fused=DEVICE.type == "cuda",
)

total_steps = len(train_loader) * cfg.num_epochs // cfg.grad_accum

def get_lr(step):
    if step < cfg.warmup_steps:
        return cfg.learning_rate * step / cfg.warmup_steps
    r = (step - cfg.warmup_steps) / max(1, total_steps - cfg.warmup_steps)
    return cfg.learning_rate * 0.5 * (1 + math.cos(math.pi * r))

# resume
start_epoch = global_step = 0
ckpts = sorted(Path(cfg.checkpoint_dir).glob("step-*.pt"))
if ckpts:
    ck = torch.load(ckpts[-1], map_location=DEVICE)
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    start_epoch, global_step = ck["epoch"], ck["step"]
    notify(f"Resumed from step {global_step}", tags="repeat")

scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")
notify(f"Training started: {cfg.num_epochs} epochs | {total_steps:,} steps | {model.num_params()/1e6:.0f}M params",
       priority="high", tags="rocket")
print(f"Total optimizer steps: {total_steps:,}")

In [ ]:
@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for i, (x, y) in enumerate(val_loader):
        if i >= 50: break
        with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16):
            _, loss = model(x.to(DEVICE), y.to(DEVICE))
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)

def save_checkpoint(epoch, step):
    p = f"{cfg.checkpoint_dir}/step-{step:07d}.pt"
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "epoch": epoch, "step": step}, p)
    for old in sorted(Path(cfg.checkpoint_dir).glob("step-*.pt"))[:-3]:
        old.unlink()

model.train()
t0 = time.time()
best_val = float("inf")

try:
    for epoch in range(start_epoch, cfg.num_epochs):
        optimizer.zero_grad()

        for local_step, (x, y) in enumerate(train_loader):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)

            with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16):
                _, loss = model(x, y)
                loss    = loss / cfg.grad_accum

            scaler.scale(loss).backward()

            if (local_step + 1) % cfg.grad_accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                lr = get_lr(global_step)
                for pg in optimizer.param_groups: pg["lr"] = lr
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1

                if global_step % cfg.log_interval == 0:
                    elapsed = time.time() - t0
                    tok_s   = cfg.batch_size * cfg.max_seq_len * cfg.log_interval / elapsed
                    print(f"ep {epoch+1} | step {global_step:,} | loss {loss.item()*cfg.grad_accum:.4f} "
                          f"| lr {lr:.2e} | {tok_s/1e3:.1f}k tok/s")
                    t0 = time.time()

                if global_step % cfg.eval_interval == 0:
                    val_loss = evaluate()
                    ppl      = math.exp(val_loss)
                    is_best  = val_loss < best_val
                    if is_best: best_val = val_loss
                    msg = (f"Step {global_step:,} | val loss {val_loss:.4f} | "
                           f"perplexity {ppl:.1f}" + (" NEW BEST" if is_best else ""))
                    print(f"  -- {msg}")
                    notify(msg, tags="bar_chart" if not is_best else "trophy")

                if global_step % cfg.save_interval == 0:
                    save_checkpoint(epoch, global_step)

        val_loss = evaluate()
        msg = f"Epoch {epoch+1}/{cfg.num_epochs} done | val loss {val_loss:.4f} | perplexity {math.exp(val_loss):.1f}"
        print(f"\n=== {msg} ===\n")
        notify(msg, priority="high", tags="checkered_flag")
        save_checkpoint(epoch + 1, global_step)

    notify("Training complete! All epochs done.", priority="urgent", tags="tada")
    print("Training complete!")

except Exception as e:
    notify(f"CRASH: {e}", priority="urgent", tags="fire")
    raise

## 9 · Generate Text

In [ ]:
def generate(prompt, max_new_tokens=150, temperature=0.8):
    ids = torch.tensor([[BOS_ID] + tokenizer.encode(prompt).ids], device=DEVICE)
    out = model.generate(ids, max_new_tokens, temperature)
    return tokenizer.decode(out[0].tolist()[len(ids[0]):])

for prompt in ["Türkiye'nin başkenti", "Atatürk, 1923 yılında", "İstanbul Boğazı"]:
    print(f"Prompt : {prompt}")
    print(f"Egemen : {prompt}{generate(prompt)}")
    print("-" * 60)

## 10 · Save & Push to Hugging Face

In [ ]:
import json, shutil

out = Path("./egemen-final")
out.mkdir(exist_ok=True)
torch.save(model.state_dict(), out / "model.pt")
(out / "config.json").write_text(json.dumps(cfg.__dict__, indent=2))
shutil.copy(tok_file, out / "tokenizer.json")
print(f"Saved to {out}")

if cfg.hf_repo:
    from huggingface_hub import login, HfApi
    login(token=os.environ.get("HUGGINGFACE_TOKEN", ""))
    api = HfApi()
    api.create_repo(cfg.hf_repo, repo_type="model", exist_ok=True)
    api.upload_folder(folder_path=str(out), repo_id=cfg.hf_repo, repo_type="model")
    notify(f"Model pushed to huggingface.co/{cfg.hf_repo}", priority="urgent", tags="hugging_face")
    print(f"Pushed to https://huggingface.co/{cfg.hf_repo}")